# 随机数与矩阵运算：NumPy 的进阶玩法

> 这是「数据分析从入门到精通」系列的第 6 篇，也是 NumPy 阶段的收官篇。这篇聊两件事：用随机数做蒙特卡洛模拟，以及矩阵运算怎么跟推荐系统搭上关系。

---

嗨，我是小荷。

NumPy 这一块我们已经走了五篇——从"为什么快"到"怎么创建数组"、"怎么取数"、"广播"、"统计函数"，今天是压轴场。

聊什么？**随机数和矩阵运算**。

你可能觉得这俩有点"高深"——随机数不就是 random 吗？矩阵运算不是线性代数的事吗？别急，听我说完，你会发现它们离实际应用一点都不远：蒙特卡洛模拟可以估算 π，矩阵乘法是推荐系统、深度学习的底层引擎。

萧何曾经说过（好吧其实没说过，但我替他说）：**运筹帷幄，随机应变**——这句话非常适合这篇文章的主题哈哈。

---

## 一、随机数：不止是"生成乱数"

### 随机数生成器基础

In [2]:
import numpy as np

# 设置随机种子——保证结果可复现（调试必备！）
np.random.seed(42)

# 均匀分布 [0, 1) 之间的随机浮点数
np.random.rand(3, 4)          # 3×4 的随机矩阵

# 标准正态分布（均值0，标准差1）
np.random.randn(3, 4)

# 指定范围的随机整数
np.random.randint(1, 100, size=(3, 4))  # [1, 100) 之间

# 指定均值和标准差的正态分布
np.random.normal(loc=50, scale=10, size=1000)   # 均值50，标准差10

# 泊松分布（模拟单位时间内事件发生次数）
np.random.poisson(lam=5, size=100)

# 二项分布（n次实验中成功k次的概率）
np.random.binomial(n=10, p=0.3, size=100)


array([1, 2, 4, 4, 4, 3, 3, 1, 2, 3, 1, 2, 3, 4, 7, 5, 3, 3, 3, 4, 1, 3,
       4, 3, 3, 3, 2, 2, 2, 0, 0, 4, 2, 3, 2, 5, 2, 3, 5, 4, 2, 4, 3, 3,
       1, 1, 4, 3, 3, 4, 4, 2, 3, 1, 4, 4, 6, 1, 1, 2, 2, 2, 5, 2, 2, 4,
       3, 4, 1, 3, 1, 1, 5, 1, 3, 3, 2, 5, 0, 4, 6, 3, 5, 1, 3, 2, 4, 6,
       2, 4, 2, 3, 3, 6, 3, 2, 1, 2, 2, 2])

> 💡 **为什么要设随机种子？** 因为 `random` 生成的是"伪随机数"，相同种子保证每次结果一样，方便调试和复现实验。写教程、写报告的时候一定要加，不然你截图和别人跑出来的结果对不上，很尴尬。

---

### 实用工具函数

In [3]:
data = np.array([1, 2, 3, 4, 5, 6, 7, 8])

# 随机打乱（shuffle 是原地操作）
np.random.shuffle(data)
print(data)  # [3 1 7 4 2 8 5 6]（结果随机）

[2 3 5 4 8 1 6 7]


In [4]:
# 随机抽样（不改变原数组）
sample = np.random.choice(data, size=3, replace=False)  # 不放回抽3个

In [5]:
# 带权重的随机抽样
items = ['A', 'B', 'C']
probs = [0.1, 0.3, 0.6]  # A概率10%，B30%，C60%
np.random.choice(items, size=10, p=probs)

array(['C', 'C', 'B', 'C', 'C', 'B', 'C', 'B', 'A', 'C'], dtype='<U1')

---

## 二、蒙特卡洛模拟：用随机数估算 π

蒙特卡洛方法的核心思想：**用大量随机实验来估算一个确定性的结果**。

**原理**：在一个边长为 2 的正方形里，内切一个半径为 1 的圆。往正方形里随机撒点，**落在圆内的点占总点数的比例 ≈ π/4**。

In [6]:
import numpy as np

def estimate_pi(n_samples):
    """用蒙特卡洛方法估算 π"""
    np.random.seed(42)

    # 在 [-1, 1] 范围内随机生成 n_samples 个点
    x = np.random.uniform(-1, 1, n_samples)
    y = np.random.uniform(-1, 1, n_samples)

    # 判断每个点是否在圆内（距离原点 <= 1）
    inside = (x**2 + y**2) <= 1.0

    # 圆内点数 / 总点数 ≈ π/4
    pi_estimate = 4 * inside.sum() / n_samples
    return pi_estimate

# 不同样本量的估算精度
for n in [100, 1000, 10000, 1000000]:
    pi_est = estimate_pi(n)
    print(f"样本量 {n:>8,}: π ≈ {pi_est:.6f}  误差: {abs(pi_est - np.pi):.6f}")


样本量      100: π ≈ 3.280000  误差: 0.138407
样本量    1,000: π ≈ 3.096000  误差: 0.045593
样本量   10,000: π ≈ 3.134800  误差: 0.006793
样本量 1,000,000: π ≈ 3.139892  误差: 0.001701


输出：
```
样本量      100: π ≈ 3.160000  误差: 0.018407
样本量    1,000: π ≈ 3.132000  误差: 0.009593
样本量   10,000: π ≈ 3.141200  误差: 0.000393
样本量 1,000,000: π ≈ 3.141788  误差: 0.000195
```

**样本量越大，估算越准** —— 这就是大数定律的直观体现。

---

## 三、矩阵运算：从加减乘到线性代数

### 基础矩阵运算

In [8]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

# 元素级运算（对应位置相乘，不是矩阵乘法！）
print(A * B)
# [[ 5 12]
#  [21 32]]

[[ 5 12]
 [21 32]]


In [9]:
# 矩阵乘法（dot product）——这才是线性代数里的矩阵乘
print(A @ B)          # 推荐写法（Python 3.5+）
print(np.dot(A, B))   # 等价写法
# [[19 22]
#  [43 50]]

[[19 22]
 [43 50]]
[[19 22]
 [43 50]]


> ⚠️ **超级常见的坑**：`A * B` 是**元素级乘法**，`A @ B` 才是**矩阵乘法**。在深度学习代码里混淆这两个，结果天差地别，记清楚！

### 矩阵的基本操作

In [10]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

# 转置
print(A.T)
# [[1 4]
#  [2 5]
#  [3 6]]

[[1 4]
 [2 5]
 [3 6]]


In [11]:
# 行列式（方阵）
square = np.array([[1, 2], [3, 4]])
print(np.linalg.det(square))   # -2.0

-2.0000000000000004


In [12]:
# 逆矩阵
print(np.linalg.inv(square))
# [[-2.   1. ]
#  [ 1.5 -0.5]]

[[-2.   1. ]
 [ 1.5 -0.5]]


In [13]:
# 特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(square)
print(f"特征值: {eigenvalues}")

特征值: [-0.37228132  5.37228132]


---

## 四、实战：矩阵运算在推荐系统里的应用

推荐系统最核心的一个思路叫**协同过滤**，本质上就是矩阵运算。

**场景**：有 4 个用户、5 部电影，已有部分评分，要预测未评分的部分。

In [14]:
import numpy as np

# 用户-电影评分矩阵（0 表示未评分）
ratings = np.array([
    [5, 3, 0, 1, 4],   # 用户A
    [4, 0, 4, 1, 2],   # 用户B
    [1, 1, 0, 5, 0],   # 用户C
    [0, 0, 5, 4, 0],   # 用户D
], dtype=float)

# 计算用户相似度（余弦相似度）
def cosine_similarity(matrix):
    """计算行向量之间的余弦相似度"""
    # 每行的范数
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    # 归一化
    normalized = matrix / (norms + 1e-8)
    # 相似度矩阵
    return normalized @ normalized.T

sim_matrix = cosine_similarity(ratings)
print("用户相似度矩阵:")
print(np.round(sim_matrix, 3))
# [[1.    0.754 0.294 0.263]
#  [0.754 1.    0.22  0.468]
#  [0.294 0.22  1.    0.627]
#  [0.263 0.468 0.627 1.   ]]

# 找出与用户A最相似的用户
user_a_sim = sim_matrix[0]
most_similar = np.argsort(user_a_sim)[::-1][1]  # 排除自身
print(f"\n与用户A最相似的是: 用户{['A','B','C','D'][most_similar]}")
print("推荐策略：把用户B喜欢但用户A未看过的电影推荐给A")


用户相似度矩阵:
[[1.    0.668 0.35  0.087]
 [0.668 1.    0.285 0.616]
 [0.35  0.285 1.    0.601]
 [0.087 0.616 0.601 1.   ]]

与用户A最相似的是: 用户B
推荐策略：把用户B喜欢但用户A未看过的电影推荐给A


**你看，推荐系统的核心就是矩阵乘法。** 后面讲机器学习的时候还会遇到更复杂的矩阵操作，现在打好基础很重要。

---

## 五、线性方程组求解

数据分析里有时候要解方程，NumPy 直接搞定：

In [15]:
# 解方程组：
# 2x + y = 8
# x + 3y = 9

A = np.array([[2, 1],
              [1, 3]])
b = np.array([8, 9])

x = np.linalg.solve(A, b)
print(f"解：x = {x[0]:.1f}, y = {x[1]:.1f}")
# 解：x = 3.0, y = 2.0

# 验证
print(f"验证: {A @ x}")  # [8. 9.]


解：x = 3.0, y = 2.0
验证: [8. 9.]


---

## 本篇小结

| 功能 | 函数 | 用途 |
|------|------|------|
| 均匀随机数 | `np.random.rand()` | 蒙特卡洛模拟 |
| 正态随机数 | `np.random.randn()` | 生成模拟数据 |
| 随机种子 | `np.random.seed()` | 结果可复现 |
| 矩阵乘法 | `A @ B` | 线性变换、神经网络 |
| 转置 | `A.T` | 数据重组 |
| 逆矩阵 | `np.linalg.inv()` | 解方程组 |
| 特征值 | `np.linalg.eig()` | PCA 降维 |

---

## 课后练习

In [20]:
import numpy as np
np.random.seed(2024)

# 任务1：用蒙特卡洛模拟，估算积分 ∫₀¹ x² dx 的值（答案应接近1/3）
# 提示：在 [0,1] 范围内随机生成 n 个 x，计算 x² 的均值即为近似积分值

# 任务2：用矩阵运算实现向量的余弦相似度
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])
# 计算 v1 和 v2 的余弦相似度，不能用循环，用矩阵运算实现


In [21]:
# 任务1：用蒙特卡洛模拟，估算积分 ∫₀¹ x² dx 的值（答案应接近1/3）
x = np.random.uniform(0, 1, n)   # 在 [0,1] 均匀撒 n 个点
result = np.mean(x ** 2)          # x² 的期望值 ≈ 积分值

print("=== 任务1：蒙特卡洛积分 ===")
print(f"模拟结果：{result:.6f}")
print(f"精确答案：{1/3:.6f}")
print(f"误差：    {abs(result - 1/3):.6f}")

# 用不同 n 看收敛
print("\n不同样本量的精度：")
for n_test in [100, 1_000, 10_000, 100_000, 1_000_000]:
    x_test = np.random.uniform(0, 1, n_test)
    est = np.mean(x_test ** 2)
    print(f"  n={n_test:>9,}：{est:.6f}（误差 {abs(est-1/3):.6f}）")


=== 任务1：蒙特卡洛积分 ===
模拟结果：0.332868
精确答案：0.333333
误差：    0.000465

不同样本量的精度：
  n=      100：0.307248（误差 0.026085）
  n=    1,000：0.319445（误差 0.013889）
  n=   10,000：0.334973（误差 0.001640）
  n=  100,000：0.332733（误差 0.000600）
  n=1,000,000：0.333075（误差 0.000258）


In [22]:
# 任务2：用矩阵运算实现向量的余弦相似度
dot     = v1 @ v2                              # 点积
norm_v1 = np.linalg.norm(v1)                  # ||v1||
norm_v2 = np.linalg.norm(v2)                  # ||v2||
cosine  = dot / (norm_v1 * norm_v2)           # cos θ

print("\n=== 任务2：余弦相似度 ===")
print(f"点积：      {dot}")
print(f"||v1||：    {norm_v1:.4f}")
print(f"||v2||：    {norm_v2:.4f}")
print(f"余弦相似度：{cosine:.6f}")


=== 任务2：余弦相似度 ===
点积：      32
||v1||：    3.7417
||v2||：    8.7750
余弦相似度：0.974632


 > 小荷提示：不知道余弦相似度是什么？请戳：[余弦相似度参考文献](https://www.ibm.com/cn-zh/think/topics/cosine-similarity)

把你的答案扔评论区，我看 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 7 篇：Pandas 入门：Series 与 DataFrame**
>
> NumPy 阶段收官！下篇开始进入 Pandas，认识数据表的两种核心形态——Series 和 DataFrame。真实的数据分析工作，80% 的时间在这里。

---

**NumPy 六篇走完，感觉怎么样？**

👇 点「在看」，推给想打好 NumPy 基础的人  
💬 评论区说说你觉得哪篇最难理解  
⭐ 关注公众号，Pandas 阶段精彩继续，萧何迷妹不鸽更

---

*「数据分析从入门到精通」系列 · 第 6 篇*  
*上一篇：[第 5 篇：NumPy 常用统计函数]*  
*下一篇：第 7 篇：Pandas 入门 — Series 与 DataFrame*